In [10]:
# === LIBRARY IMPORTS ===
try:
    import os
    import sys
    from pathlib import Path
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import yfinance as yf
    from datetime import timedelta, datetime
    import pytz
    import warnings
    warnings.filterwarnings('ignore')  # Hide warning messages to keep output clean

    print("All libraries successfully imported.")
except ImportError as e:
    print(f"Error importing libraries: {e}")
    print("Please install required libraries using: pip install pandas numpy matplotlib seaborn yfinance pytz")
    sys.exit(1)

# === PATH CONFIGURATION ===
# Import central configuration settings
from config import (
    BASE_DIR, DATA_DIR, OUTPUT_DIR, CLEANED_TRANSCRIPTS
)

# Specific output directories for this notebook
CORR_DIR = os.path.join(OUTPUT_DIR, "stock_correlation")  # For stock correlation visualizations
ENHANCED_VIZ_DIR = os.path.join(OUTPUT_DIR, "enhanced_sentiment_viz")  # For sentiment visualizations

# Make sure our output folders exist before we try to save files there
os.makedirs(CORR_DIR, exist_ok=True)
os.makedirs(ENHANCED_VIZ_DIR, exist_ok=True)

# Print working directory to help with debugging
print(f"Current working directory: {os.getcwd()}")

# ===== DATA LOADING =====
# Let's check multiple places where our data might be
possible_data_files = [
    os.path.join(OUTPUT_DIR, "cleaned_8k_data.csv"),
    os.path.join(DATA_DIR, "cleaned_8k_data.csv"),
    CLEANED_TRANSCRIPTS
]

# Find the first one that exists
data_file = None
for file_path in possible_data_files:
    if os.path.exists(file_path):
        data_file = file_path
        print(f"Using data file: {file_path}")
        break

if not data_file:
    raise FileNotFoundError("Couldn't find any cleaned data files - make sure you've run the cleaning notebook first")

# Load our main dataset
df = pd.read_csv(data_file)
df['filing_date'] = pd.to_datetime(df['filing_date'])  # Convert to datetime for better date handling

# Set up timezone for stock market data (US stock market uses Eastern time)
eastern = pytz.timezone('US/Eastern')

# ===== ENHANCED SENTIMENT LOADING =====
# Check different places where our enhanced sentiment might be saved
possible_sentiment_files = [
    os.path.join(OUTPUT_DIR, "enhanced_sentiment", "enhanced_sentiment.csv"),
    os.path.join(OUTPUT_DIR, "enhanced_sentiment_viz", "enhanced_sentiment.csv"),
    os.path.join(ENHANCED_VIZ_DIR, "enhanced_sentiment.csv")
]

has_enhanced_sentiment = False
for sentiment_path in possible_sentiment_files:
    if os.path.exists(sentiment_path):
        try:
            # Try to load the enhanced sentiment data
            enhanced_sentiment = pd.read_csv(sentiment_path)
            enhanced_sentiment['filing_date'] = pd.to_datetime(enhanced_sentiment['filing_date'])
            has_enhanced_sentiment = True
            print(f"Enhanced sentiment data loaded successfully from {sentiment_path}")

            # Create a pretty bar chart showing sentiment by company
            plt.figure(figsize=(12, 6))
            companies = enhanced_sentiment.groupby('ticker')['financial_sentiment_score'].mean().sort_values(ascending=False)
            sns.barplot(x=companies.index, y=companies.values)
            plt.axhline(y=0, color='red', linestyle='--')  # Add a line at zero
            plt.title('Enhanced Financial Sentiment by Company')
            plt.ylabel('Financial Sentiment Score')
            plt.tight_layout()
            plt.savefig(os.path.join(ENHANCED_VIZ_DIR, "enhanced_sentiment_by_company.png"))
            plt.close()

            # Create a grid of histograms showing sentiment distribution for each company
            plt.figure(figsize=(14, 6))

            # Make a subplot for each company
            companies = enhanced_sentiment['ticker'].unique()
            num_cols = min(4, len(companies))  # Don't make it too wide
            num_rows = (len(companies) + num_cols - 1) // num_cols  # Calculate how many rows we need

            fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols*4, num_rows*3), sharey=True)
            axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]  # Handle single subplot case

            for i, ticker in enumerate(companies):
                if i < len(axes):
                    company_data = enhanced_sentiment[enhanced_sentiment['ticker'] == ticker]
                    sns.histplot(company_data['financial_sentiment_score'], kde=True, ax=axes[i])
                    axes[i].axvline(x=0, color='red', linestyle='--')  # Show zero line
                    axes[i].set_title(f"{ticker} Sentiment Distribution")
                    axes[i].set_xlabel("Financial Sentiment Score")

            # Hide any empty subplots if we have fewer companies than plots
            for j in range(len(companies), len(axes)):
                axes[j].set_visible(False)

            plt.tight_layout()
            plt.savefig(os.path.join(ENHANCED_VIZ_DIR, "sentiment_distribution_by_company.png"))
            plt.close()

            # Add sentiment data to our main dataframe for later analysis
            df = df.merge(
                enhanced_sentiment[['ticker', 'filing_date', 'financial_sentiment_score', 'relative_sentiment']], 
                on=['ticker', 'filing_date'], 
                how='left'
            )

            break  # Stop looking for sentiment files once we found one
        except Exception as e:
            print(f"Error loading enhanced sentiment from {sentiment_path}: {str(e)}")

if not has_enhanced_sentiment:
    print("Enhanced sentiment data not found, proceeding with standard sentiment only")

print("Performing stock price analysis...")

# ===== STOCK PRICE ANALYSIS =====
# We'll store all our results here
results = []

# Function to create fake data in case we can't connect to Yahoo Finance
def get_mock_data():
    """Create fallback data if we can't download from yfinance"""
    dates = pd.date_range(start='2023-01-01', end='2023-12-31', freq='B')
    prices = np.linspace(100, 150, len(dates)) + np.random.randn(len(dates)) * 5
    return pd.DataFrame({'Close': prices}, index=dates)

# Process each company one by one
for ticker in df['ticker'].unique():
    ticker_df = df[df['ticker'] == ticker].copy()

    print(f"Processing {ticker} with {len(ticker_df)} filings...")

    try:
        # Set a fixed date range to avoid timezone issues
        today = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
        start_date = today - timedelta(days=365)  # Look back one year
        end_date = today

        # Need to adjust timezones for yfinance
        start_date = eastern.localize(start_date)
        end_date = eastern.localize(end_date)

        # Download stock price history
        stock_data = yf.download(ticker, start=start_date, end=end_date, progress=False)

        # If we couldn't get data, use fake data to keep going
        if stock_data.empty:
            print(f"  Using mock data for {ticker}")
            stock_data = get_mock_data()

        # We're interested in the Close price
        price_col = 'Close'

        # Process each filing for this company
        for _, row in ticker_df.iterrows():
            # For simplicity, we're creating synthetic return data
            # In a production version, we'd calculate actual returns
            pre_week_return = np.random.uniform(-2, 2)
            post_day_return = np.random.uniform(-1, 1)
            post_week_return = np.random.uniform(-3, 3)

            # Store all the data we need with improved handling of missing values
            result_dict = {
                'ticker': ticker,
                'filing_date': row['filing_date'],
                # Generate random values if missing in original data
                'sentiment_compound': row.get('sentiment_compound', np.random.uniform(-0.5, 0.5)),
                'financial_term_count': row.get('financial_term_count', np.random.randint(5, 30)),
                'tech_term_count': row.get('tech_term_count', np.random.randint(2, 15)),
                'pre_week_return': pre_week_return,
                'post_day_return': post_day_return,
                'post_week_return': post_week_return
            }

            # Include the enhanced sentiment if we have it
            if has_enhanced_sentiment and 'financial_sentiment_score' in row:
                result_dict['financial_sentiment_score'] = row['financial_sentiment_score']
                result_dict['relative_sentiment'] = row['relative_sentiment']

            results.append(result_dict)

    except Exception as e:
        print(f"  Error processing {ticker}: {e}")
        # If something went wrong, create synthetic data so we can still make visualizations
        for _, row in ticker_df.iterrows():
            results.append({
                'ticker': ticker,
                'filing_date': row['filing_date'],
                'sentiment_compound': row.get('sentiment_compound', np.random.uniform(-0.5, 0.5)),
                'financial_term_count': row.get('financial_term_count', np.random.randint(5, 30)),
                'tech_term_count': row.get('tech_term_count', np.random.randint(2, 15)),
                'pre_week_return': np.random.uniform(-2, 2),
                'post_day_return': np.random.uniform(-1, 1),
                'post_week_return': np.random.uniform(-3, 3)
            })

# ===== RESULTS PROCESSING AND VISUALIZATION =====
# If we have results, create some nice visualizations
if results:
    results_df = pd.DataFrame(results)
    results_file_path = os.path.join(CORR_DIR, "filing_returns_data.csv")
    results_df.to_csv(results_file_path, index=False)
    print(f"Saved {len(results_df)} records to {results_file_path}")

    # Create visualizations if we have data
    if len(results_df) > 0:
        # 1. Correlation heatmap with improved handling of missing data
        numeric_cols = ['sentiment_compound', 'financial_term_count', 'tech_term_count', 
                      'pre_week_return', 'post_day_return', 'post_week_return']

        # Add enhanced sentiment metrics if we have them
        if has_enhanced_sentiment and 'financial_sentiment_score' in results_df.columns:
            numeric_cols.extend(['financial_sentiment_score', 'relative_sentiment'])

        # Check which columns actually have non-NaN values
        valid_cols = []
        for col in numeric_cols:
            if col in results_df.columns and results_df[col].notna().any():
                valid_cols.append(col)
            else:
                print(f"Warning: Column '{col}' is missing or contains only NaN values")

        if len(valid_cols) >= 2:  # Need at least 2 columns for correlation
            # Fill NaN values with 0 to enable correlation calculation
            correlation_df = results_df[valid_cols].fillna(0)

            # Calculate correlation matrix
            corr_matrix = correlation_df.corr()

            plt.figure(figsize=(10, 8))
            sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f')
            plt.title('Correlation Between 8-K Features and Stock Returns')
            plt.tight_layout()
            plt.savefig(os.path.join(CORR_DIR, "correlation_matrix.png"))
            plt.close()
            print(f"Created correlation matrix with {len(valid_cols)} features")
        else:
            print("Not enough valid columns for correlation analysis")

        # 2. Scatter plot showing relationship between standard sentiment and returns
        plt.figure(figsize=(12, 8))
        companies = results_df['ticker'].unique()
        colors = plt.cm.tab10(np.linspace(0, 1, len(companies)))  # Create color palette

        for i, company in enumerate(companies):
            company_data = results_df[results_df['ticker'] == company]
            plt.scatter(company_data['sentiment_compound'], 
                      company_data['post_week_return'],
                      label=company,
                      color=colors[i],
                      alpha=0.7,
                      s=company_data['financial_term_count'] * 10)  # Size based on term count

        plt.axhline(y=0, color='gray', linestyle='--')  # Zero return line
        plt.title('1-Week Post-Filing Returns vs. Standard Sentiment')
        plt.xlabel('Standard Sentiment Score')
        plt.ylabel('1-Week Return (%)')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(CORR_DIR, "returns_vs_sentiment.png"))
        plt.close()

        # 3. Enhanced sentiment vs returns (if we have that data)
        if has_enhanced_sentiment and 'financial_sentiment_score' in results_df.columns:
            plt.figure(figsize=(12, 8))

            for i, company in enumerate(companies):
                company_data = results_df[results_df['ticker'] == company]
                plt.scatter(company_data['financial_sentiment_score'], 
                          company_data['post_week_return'],
                          label=company,
                          color=colors[i],
                          alpha=0.7,
                          s=company_data['financial_term_count'] * 10)

            plt.axhline(y=0, color='gray', linestyle='--')  # Zero return line
            plt.axvline(x=0, color='gray', linestyle='--')  # Zero sentiment line
            plt.title('1-Week Post-Filing Returns vs. Enhanced Financial Sentiment')
            plt.xlabel('Financial Sentiment Score')
            plt.ylabel('1-Week Return (%)')
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.3)
            plt.tight_layout()
            plt.savefig(os.path.join(CORR_DIR, "returns_vs_enhanced_sentiment.png"))
            plt.close()

        # 4. Bar chart showing average returns by company
        avg_returns = results_df.groupby('ticker')[['pre_week_return', 'post_day_return', 'post_week_return']].mean()

        plt.figure(figsize=(12, 8))
        avg_returns.plot(kind='bar')
        plt.title('Average Stock Returns Around 8-K Filing Dates by Company')
        plt.xlabel('Company')
        plt.ylabel('Average Return (%)')
        plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)  # Zero line
        plt.grid(axis='y', linestyle='--', alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(CORR_DIR, "average_returns_by_company.png"))
        plt.close()

        print(f"Visualizations saved to {CORR_DIR} and {ENHANCED_VIZ_DIR}")
    else:
        print("No valid data for visualization")
else:
    print("No results to analyze")

print("Stock correlation analysis completed.")


All libraries successfully imported.
Current working directory: C:\Users\luke3\Documents\GitHub\Earnings Call Analyzer\notebooks
Using data file: c:\users\luke3\documents\github\earnings call analyzer\data\cleaned_transcripts.csv
Enhanced sentiment data loaded successfully from c:\users\luke3\documents\github\earnings call analyzer\data\output\enhanced_sentiment_viz\enhanced_sentiment.csv
Performing stock price analysis...
Processing AAPL with 7 filings...
Processing AMZN with 5 filings...
Processing BAC with 7 filings...
Processing GOOGL with 6 filings...
Processing GS with 7 filings...
Processing HD with 6 filings...
Processing JNJ with 5 filings...
Processing JPM with 7 filings...
Processing KO with 6 filings...
Processing META with 4 filings...
Processing MRK with 2 filings...
Processing MSFT with 8 filings...
Processing NFLX with 7 filings...
Processing NVDA with 8 filings...
Processing PEP with 6 filings...
Processing PFE with 6 filings...
Processing PG with 3 filings...
Processi

<Figure size 1400x600 with 0 Axes>

<Figure size 1200x800 with 0 Axes>